# Café Business Analysis: Volatility-Profitability Matrix

## Executive Summary
This analysis constructs a volatility-profitability matrix to restructure the café's menu portfolio, incorporating temperature, holiday, weekend, and school-break effects to maximize profit per hour.

**Key Question:** How can we restructure the café's entire menu portfolio using a volatility–profitability matrix derived from item-level sales, temperature shifts, holiday effects, and school-break behavior, to design a predictive operational model that maximizes profit per hour?

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

## 1. Data Loading & Exploration

In [ ]:
# Load datasets
transactions = pd.read_csv('Dataset/Cafe Transaction store.csv')
date_info = pd.read_csv('Dataset/Cafe DateInfo.csv')
metadata = pd.read_csv('Dataset/Cafe Sell MetaData.csv')

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nTransactions: {len(transactions):,} rows")
print(f"Date Info: {len(date_info):,} rows")
print(f"Metadata: {len(metadata):,} rows")

print("\n" + "=" * 80)
print("TRANSACTION DATA SAMPLE")
print("=" * 80)
print(transactions.head(10))

print("\n" + "=" * 80)
print("METADATA - PRODUCT STRUCTURE")
print("=" * 80)
print(metadata)

## 2. Data Preparation & Feature Engineering

In [ ]:
# Merge transactions with date info
df = transactions.merge(date_info, on='CALENDAR_DATE', how='left')

# Parse dates
df['DATE'] = pd.to_datetime(df['CALENDAR_DATE'], format='%m/%d/%y', errors='coerce')
df['MONTH'] = df['DATE'].dt.month
df['QUARTER'] = df['DATE'].dt.quarter
df['DAY_OF_YEAR'] = df['DATE'].dt.dayofyear

# Calculate revenue
df['REVENUE'] = df['PRICE'] * df['QUANTITY']

# Create temperature bins
df['TEMP_BIN'] = pd.cut(df['AVERAGE_TEMPERATURE'], 
                         bins=[0, 32, 50, 70, 90], 
                         labels=['Cold (<32°F)', 'Cool (32-50°F)', 'Moderate (50-70°F)', 'Hot (70°F+)'])

# Handle combo products - expand to individual items
combo_items = []
for idx, row in df[df['SELL_CATEGORY'] == 2].iterrows():
    combo_products = metadata[metadata['SELL_ID'] == row['SELL_ID']]
    for _, product in combo_products.iterrows():
        combo_items.append({
            'DATE': row['DATE'],
            'ITEM_NAME': product['ITEM_NAME'],
            'QUANTITY': row['QUANTITY'],
            'REVENUE': row['REVENUE'] / len(combo_products),
            'PRICE': row['PRICE'] / len(combo_products),
            'SELL_CATEGORY': 'COMBO',
            'IS_WEEKEND': row['IS_WEEKEND'],
            'IS_SCHOOLBREAK': row['IS_SCHOOLBREAK'],
            'AVERAGE_TEMPERATURE': row['AVERAGE_TEMPERATURE'],
            'IS_OUTDOOR': row['IS_OUTDOOR'],
            'HOLIDAY': row['HOLIDAY'],
            'TEMP_BIN': row['TEMP_BIN'],
            'MONTH': row['MONTH'],
            'QUARTER': row['QUARTER']
        })

# Single items
single_items = []
for idx, row in df[df['SELL_CATEGORY'] == 0].iterrows():
    product = metadata[metadata['SELL_ID'] == row['SELL_ID']].iloc[0]
    single_items.append({
        'DATE': row['DATE'],
        'ITEM_NAME': product['ITEM_NAME'],
        'QUANTITY': row['QUANTITY'],
        'REVENUE': row['REVENUE'],
        'PRICE': row['PRICE'],
        'SELL_CATEGORY': 'SINGLE',
        'IS_WEEKEND': row['IS_WEEKEND'],
        'IS_SCHOOLBREAK': row['IS_SCHOOLBREAK'],
        'AVERAGE_TEMPERATURE': row['AVERAGE_TEMPERATURE'],
        'IS_OUTDOOR': row['IS_OUTDOOR'],
        'HOLIDAY': row['HOLIDAY'],
        'TEMP_BIN': row['TEMP_BIN'],
        'MONTH': row['MONTH'],
        'QUARTER': row['QUARTER']
    })

# Combine all items
item_df = pd.DataFrame(combo_items + single_items)

print("\n" + "=" * 80)
print("ITEM-LEVEL DATA PREPARED")
print("=" * 80)
print(f"Total item-level records: {len(item_df):,}")
print(f"\nItems breakdown:")
print(item_df['ITEM_NAME'].value_counts())
print(f"\nSales channel breakdown:")
print(item_df['SELL_CATEGORY'].value_counts())

## 3. Profitability Analysis

In [ ]:
# Calculate profitability metrics by item
profitability = item_df.groupby('ITEM_NAME').agg({
    'REVENUE': 'sum',
    'QUANTITY': 'sum',
    'PRICE': 'mean'
}).reset_index()

profitability['AVG_REVENUE_PER_UNIT'] = profitability['REVENUE'] / profitability['QUANTITY']
profitability['PROFIT_MARGIN'] = profitability['AVG_REVENUE_PER_UNIT'] / profitability['PRICE']  # Simplified

# Calculate daily averages
n_days = (item_df['DATE'].max() - item_df['DATE'].min()).days
profitability['REVENUE_PER_DAY'] = profitability['REVENUE'] / n_days
profitability['UNITS_PER_DAY'] = profitability['QUANTITY'] / n_days

print("\n" + "=" * 80)
print("PROFITABILITY METRICS BY ITEM")
print("=" * 80)
print(profitability.sort_values('REVENUE', ascending=False).to_string(index=False))

# Visualize profitability
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Total revenue by item
axes[0, 0].bar(profitability['ITEM_NAME'], profitability['REVENUE'], color='steelblue')
axes[0, 0].set_title('Total Revenue by Item (2012-2015)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Revenue ($)', fontsize=12)
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(profitability['REVENUE']):
    axes[0, 0].text(i, v, f'${v:,.0f}', ha='center', va='bottom', fontsize=10)

# Total quantity by item
axes[0, 1].bar(profitability['ITEM_NAME'], profitability['QUANTITY'], color='coral')
axes[0, 1].set_title('Total Units Sold by Item', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Quantity', fontsize=12)
axes[0, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(profitability['QUANTITY']):
    axes[0, 1].text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=10)

# Average price per item
axes[1, 0].bar(profitability['ITEM_NAME'], profitability['AVG_REVENUE_PER_UNIT'], color='seagreen')
axes[1, 0].set_title('Average Revenue per Unit', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Revenue per Unit ($)', fontsize=12)
axes[1, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(profitability['AVG_REVENUE_PER_UNIT']):
    axes[1, 0].text(i, v, f'${v:.2f}', ha='center', va='bottom', fontsize=10)

# Revenue per day
axes[1, 1].bar(profitability['ITEM_NAME'], profitability['REVENUE_PER_DAY'], color='purple')
axes[1, 1].set_title('Average Daily Revenue', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Revenue per Day ($)', fontsize=12)
axes[1, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(profitability['REVENUE_PER_DAY']):
    axes[1, 1].text(i, v, f'${v:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('profitability_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Volatility Analysis
Analyze sales volatility across different conditions: temperature, holidays, weekends, and school breaks.

In [ ]:
# Calculate daily sales for each item
daily_sales = item_df.groupby(['DATE', 'ITEM_NAME']).agg({
    'QUANTITY': 'sum',
    'REVENUE': 'sum'
}).reset_index()

# Calculate volatility (coefficient of variation)
volatility = daily_sales.groupby('ITEM_NAME').agg({
    'QUANTITY': ['mean', 'std']
}).reset_index()
volatility.columns = ['ITEM_NAME', 'AVG_DAILY_QTY', 'STD_DAILY_QTY']
volatility['VOLATILITY'] = (volatility['STD_DAILY_QTY'] / volatility['AVG_DAILY_QTY']) * 100

print("\n" + "=" * 80)
print("VOLATILITY METRICS (Coefficient of Variation %)")
print("=" * 80)
print(volatility.sort_values('VOLATILITY', ascending=False).to_string(index=False))

# Temperature sensitivity
temp_impact = item_df.groupby(['ITEM_NAME', 'TEMP_BIN'])['QUANTITY'].sum().unstack(fill_value=0)
temp_impact_pct = temp_impact.div(temp_impact.sum(axis=1), axis=0) * 100

print("\n" + "=" * 80)
print("TEMPERATURE SENSITIVITY (% of sales by temperature range)")
print("=" * 80)
print(temp_impact_pct.to_string())

# Weekend vs Weekday
weekend_impact = item_df.groupby(['ITEM_NAME', 'IS_WEEKEND'])['QUANTITY'].sum().unstack(fill_value=0)
weekend_impact.columns = ['Weekday', 'Weekend']
weekend_impact['Weekend_Lift_%'] = ((weekend_impact['Weekend'] - weekend_impact['Weekday']) / weekend_impact['Weekday']) * 100

print("\n" + "=" * 80)
print("WEEKEND VS WEEKDAY IMPACT")
print("=" * 80)
print(weekend_impact.to_string())

# Holiday impact
item_df['IS_HOLIDAY'] = item_df['HOLIDAY'].apply(lambda x: 0 if x == 'NULL' or pd.isna(x) else 1)
holiday_impact = item_df.groupby(['ITEM_NAME', 'IS_HOLIDAY'])['QUANTITY'].sum().unstack(fill_value=0)
holiday_impact.columns = ['Non-Holiday', 'Holiday']
holiday_impact['Holiday_Lift_%'] = ((holiday_impact['Holiday'] - holiday_impact['Non-Holiday']) / holiday_impact['Non-Holiday']) * 100

print("\n" + "=" * 80)
print("HOLIDAY IMPACT")
print("=" * 80)
print(holiday_impact.to_string())

# School break impact
school_impact = item_df.groupby(['ITEM_NAME', 'IS_SCHOOLBREAK'])['QUANTITY'].sum().unstack(fill_value=0)
school_impact.columns = ['School', 'School Break']
school_impact['SchoolBreak_Lift_%'] = ((school_impact['School Break'] - school_impact['School']) / school_impact['School']) * 100

print("\n" + "=" * 80)
print("SCHOOL BREAK IMPACT")
print("=" * 80)
print(school_impact.to_string())

## 5. Volatility-Profitability Matrix Construction

In [ ]:
# Merge profitability and volatility data
matrix_df = profitability.merge(volatility[['ITEM_NAME', 'VOLATILITY']], on='ITEM_NAME')

# Normalize metrics for matrix positioning (0-100 scale)
matrix_df['PROFITABILITY_SCORE'] = (matrix_df['REVENUE'] - matrix_df['REVENUE'].min()) / (matrix_df['REVENUE'].max() - matrix_df['REVENUE'].min()) * 100
matrix_df['VOLATILITY_SCORE'] = matrix_df['VOLATILITY']

# Define quadrants
prof_median = matrix_df['PROFITABILITY_SCORE'].median()
vol_median = matrix_df['VOLATILITY_SCORE'].median()

def classify_quadrant(row):
    if row['PROFITABILITY_SCORE'] >= prof_median and row['VOLATILITY_SCORE'] < vol_median:
        return 'Stars (High Profit, Low Volatility)'
    elif row['PROFITABILITY_SCORE'] >= prof_median and row['VOLATILITY_SCORE'] >= vol_median:
        return 'Wildcards (High Profit, High Volatility)'
    elif row['PROFITABILITY_SCORE'] < prof_median and row['VOLATILITY_SCORE'] < vol_median:
        return 'Steady (Low Profit, Low Volatility)'
    else:
        return 'Problem (Low Profit, High Volatility)'

matrix_df['QUADRANT'] = matrix_df.apply(classify_quadrant, axis=1)

print("\n" + "=" * 80)
print("VOLATILITY-PROFITABILITY MATRIX")
print("=" * 80)
print(matrix_df[['ITEM_NAME', 'REVENUE', 'VOLATILITY', 'PROFITABILITY_SCORE', 'VOLATILITY_SCORE', 'QUADRANT']].to_string(index=False))

# Visualize the matrix
plt.figure(figsize=(14, 10))

# Color map for quadrants
colors = {'Stars (High Profit, Low Volatility)': 'green',
          'Wildcards (High Profit, High Volatility)': 'gold',
          'Steady (Low Profit, Low Volatility)': 'lightblue',
          'Problem (Low Profit, High Volatility)': 'red'}

for quadrant in matrix_df['QUADRANT'].unique():
    subset = matrix_df[matrix_df['QUADRANT'] == quadrant]
    plt.scatter(subset['PROFITABILITY_SCORE'], subset['VOLATILITY_SCORE'], 
                c=colors[quadrant], label=quadrant, s=500, alpha=0.7, edgecolors='black', linewidths=2)
    
    for idx, row in subset.iterrows():
        plt.annotate(row['ITEM_NAME'], 
                    (row['PROFITABILITY_SCORE'], row['VOLATILITY_SCORE']),
                    fontsize=12, fontweight='bold', ha='center', va='center')

# Add quadrant lines
plt.axhline(y=vol_median, color='gray', linestyle='--', linewidth=2, alpha=0.5)
plt.axvline(x=prof_median, color='gray', linestyle='--', linewidth=2, alpha=0.5)

plt.xlabel('Profitability Score →', fontsize=14, fontweight='bold')
plt.ylabel('Volatility (CV%) →', fontsize=14, fontweight='bold')
plt.title('Volatility-Profitability Matrix: Menu Portfolio Analysis', fontsize=16, fontweight='bold')
plt.legend(loc='upper left', fontsize=11, frameon=True, shadow=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('volatility_profitability_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Predictive Operational Model
Build a model to predict optimal inventory and staffing based on conditions.

In [ ]:
# Aggregate daily sales with all contextual factors
daily_model_data = item_df.groupby(['DATE', 'ITEM_NAME', 'AVERAGE_TEMPERATURE', 'IS_WEEKEND', 
                                     'IS_SCHOOLBREAK', 'IS_HOLIDAY', 'IS_OUTDOOR', 'MONTH']).agg({
    'QUANTITY': 'sum',
    'REVENUE': 'sum'
}).reset_index()

# Temperature correlation
temp_corr = daily_model_data.groupby('ITEM_NAME').apply(
    lambda x: x['AVERAGE_TEMPERATURE'].corr(x['QUANTITY'])
).reset_index()
temp_corr.columns = ['ITEM_NAME', 'TEMP_CORRELATION']

print("\n" + "=" * 80)
print("TEMPERATURE CORRELATION WITH SALES")
print("=" * 80)
print(temp_corr.sort_values('TEMP_CORRELATION', ascending=False).to_string(index=False))

# Seasonal patterns
seasonal = item_df.groupby(['ITEM_NAME', 'QUARTER'])['QUANTITY'].sum().unstack(fill_value=0)
seasonal_pct = seasonal.div(seasonal.sum(axis=1), axis=0) * 100

print("\n" + "=" * 80)
print("SEASONAL DISTRIBUTION (% of annual sales by quarter)")
print("=" * 80)
print(seasonal_pct.to_string())

# Visualize seasonal patterns
fig, ax = plt.subplots(figsize=(14, 8))
seasonal_pct.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Seasonal Sales Distribution by Quarter', fontsize=16, fontweight='bold')
ax.set_xlabel('Quarter', fontsize=12)
ax.set_ylabel('% of Annual Sales', fontsize=12)
ax.legend(title='Product', fontsize=10, title_fontsize=11)
ax.set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4'], rotation=0)
plt.tight_layout()
plt.savefig('seasonal_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

# Temperature heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(temp_impact_pct, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': '% of Sales'})
plt.title('Temperature Sensitivity Heatmap', fontsize=16, fontweight='bold')
plt.xlabel('Temperature Range', fontsize=12)
plt.ylabel('Product', fontsize=12)
plt.tight_layout()
plt.savefig('temperature_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Strategic Recommendations
Based on the volatility-profitability matrix analysis.

In [ ]:
print("\n" + "=" * 80)
print("STRATEGIC RECOMMENDATIONS BY QUADRANT")
print("=" * 80)

for quadrant in sorted(matrix_df['QUADRANT'].unique()):
    items = matrix_df[matrix_df['QUADRANT'] == quadrant]['ITEM_NAME'].tolist()
    print(f"\n{quadrant}:")
    print(f"  Products: {', '.join(items)}")
    
    if 'Stars' in quadrant:
        print("  → STRATEGY: MAXIMIZE - These are your cash cows.")
        print("     • Ensure consistent availability and quality")
        print("     • Promote heavily in marketing")
        print("     • Consider premium positioning")
        print("     • Bundle with lower-performing items")
    
    elif 'Wildcards' in quadrant:
        print("  → STRATEGY: STABILIZE - High profit but unpredictable.")
        print("     • Use predictive inventory based on weather/events")
        print("     • Implement dynamic pricing")
        print("     • Run targeted promotions during slow periods")
        print("     • Study volatility drivers for better forecasting")
    
    elif 'Steady' in quadrant:
        print("  → STRATEGY: OPTIMIZE - Reliable but low margin.")
        print("     • Reduce costs through bulk purchasing")
        print("     • Use as loss leaders to drive traffic")
        print("     • Bundle with high-margin items")
        print("     • Consider slight price increases")
    
    else:  # Problem
        print("  → STRATEGY: REFORM OR REMOVE")
        print("     • Analyze if worth keeping on menu")
        print("     • Consider repositioning or reformulation")
        print("     • If removing, transition customers to Stars")
        print("     • Evaluate operational complexity vs. value")

# Calculate potential profit impact
print("\n" + "=" * 80)
print("PROFIT MAXIMIZATION OPPORTUNITIES")
print("=" * 80)

# Assume operating hours: 8 AM - 8 PM (12 hours)
hours_per_day = 12
matrix_df['REVENUE_PER_HOUR'] = matrix_df['REVENUE_PER_DAY'] / hours_per_day

print("\nCurrent Revenue Per Hour by Product:")
for idx, row in matrix_df.sort_values('REVENUE_PER_HOUR', ascending=False).iterrows():
    print(f"  {row['ITEM_NAME']}: ${row['REVENUE_PER_HOUR']:.2f}/hour ({row['QUADRANT']})")

total_revenue = matrix_df['REVENUE'].sum()
total_hours = n_days * hours_per_day
current_rph = total_revenue / total_hours

print(f"\nCurrent Overall Revenue Per Hour: ${current_rph:.2f}")
print(f"\nPotential Optimization:")
print(f"  If we increase Stars/Wildcards sales by 20% and reduce Problem items by 50%:")

stars_wildcards_revenue = matrix_df[
    matrix_df['QUADRANT'].str.contains('Stars|Wildcards')
]['REVENUE'].sum()

problem_revenue = matrix_df[
    matrix_df['QUADRANT'].str.contains('Problem')
]['REVENUE'].sum()

optimized_revenue = (stars_wildcards_revenue * 1.2) + (problem_revenue * 0.5) + \
                    matrix_df[matrix_df['QUADRANT'].str.contains('Steady')]['REVENUE'].sum()

optimized_rph = optimized_revenue / total_hours
improvement = ((optimized_rph - current_rph) / current_rph) * 100

print(f"  Projected Revenue Per Hour: ${optimized_rph:.2f}")
print(f"  Improvement: {improvement:.1f}%")
print(f"  Annual Revenue Increase: ${(optimized_revenue - total_revenue):,.2f}")

## 8. Final Summary Dashboard

In [ ]:
# Create comprehensive summary visualization
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Revenue by item
ax1 = fig.add_subplot(gs[0, :])
bars = ax1.bar(matrix_df['ITEM_NAME'], matrix_df['REVENUE'], 
               color=[colors[q] for q in matrix_df['QUADRANT']], edgecolor='black', linewidth=1.5)
ax1.set_title('Total Revenue by Item (Colored by Strategic Quadrant)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Revenue ($)', fontsize=12)
for i, (v, q) in enumerate(zip(matrix_df['REVENUE'], matrix_df['QUADRANT'])):
    ax1.text(i, v, f'${v:,.0f}\n{q.split("(")[0].strip()}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Temperature sensitivity
ax2 = fig.add_subplot(gs[1, 0])
temp_corr_sorted = temp_corr.sort_values('TEMP_CORRELATION')
colors_temp = ['red' if x < 0 else 'green' for x in temp_corr_sorted['TEMP_CORRELATION']]
ax2.barh(temp_corr_sorted['ITEM_NAME'], temp_corr_sorted['TEMP_CORRELATION'], color=colors_temp, alpha=0.7)
ax2.set_title('Temperature Sensitivity', fontsize=12, fontweight='bold')
ax2.set_xlabel('Correlation', fontsize=10)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)

# 3. Weekend lift
ax3 = fig.add_subplot(gs[1, 1])
weekend_sorted = weekend_impact.sort_values('Weekend_Lift_%')
colors_weekend = ['red' if x < 0 else 'green' for x in weekend_sorted['Weekend_Lift_%']]
ax3.barh(weekend_sorted.index, weekend_sorted['Weekend_Lift_%'], color=colors_weekend, alpha=0.7)
ax3.set_title('Weekend Effect', fontsize=12, fontweight='bold')
ax3.set_xlabel('Lift %', fontsize=10)
ax3.axvline(x=0, color='black', linestyle='-', linewidth=1)

# 4. Holiday lift
ax4 = fig.add_subplot(gs[1, 2])
holiday_sorted = holiday_impact.sort_values('Holiday_Lift_%')
colors_holiday = ['red' if x < 0 else 'green' for x in holiday_sorted['Holiday_Lift_%']]
ax4.barh(holiday_sorted.index, holiday_sorted['Holiday_Lift_%'], color=colors_holiday, alpha=0.7)
ax4.set_title('Holiday Effect', fontsize=12, fontweight='bold')
ax4.set_xlabel('Lift %', fontsize=10)
ax4.axvline(x=0, color='black', linestyle='-', linewidth=1)

# 5. Volatility comparison
ax5 = fig.add_subplot(gs[2, 0])
volatility_sorted = volatility.sort_values('VOLATILITY')
ax5.barh(volatility_sorted['ITEM_NAME'], volatility_sorted['VOLATILITY'], color='orange', alpha=0.7)
ax5.set_title('Sales Volatility (CV%)', fontsize=12, fontweight='bold')
ax5.set_xlabel('Coefficient of Variation %', fontsize=10)

# 6. Revenue per hour
ax6 = fig.add_subplot(gs[2, 1])
rph_sorted = matrix_df.sort_values('REVENUE_PER_HOUR')
ax6.barh(rph_sorted['ITEM_NAME'], rph_sorted['REVENUE_PER_HOUR'], 
         color=[colors[q] for q in rph_sorted['QUADRANT']], alpha=0.7, edgecolor='black')
ax6.set_title('Revenue Per Hour', fontsize=12, fontweight='bold')
ax6.set_xlabel('$/Hour', fontsize=10)

# 7. Strategic positioning summary
ax7 = fig.add_subplot(gs[2, 2])
quadrant_counts = matrix_df['QUADRANT'].value_counts()
wedges, texts, autotexts = ax7.pie(quadrant_counts.values, labels=None, autopct='%1.0f%%',
                                     colors=[colors[q] for q in quadrant_counts.index],
                                     startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
ax7.set_title('Portfolio Distribution', fontsize=12, fontweight='bold')
ax7.legend([q.split('(')[0].strip() for q in quadrant_counts.index], 
           loc='upper left', bbox_to_anchor=(1, 1), fontsize=9)

plt.suptitle('Café Business Intelligence Dashboard', fontsize=18, fontweight='bold', y=0.995)
plt.savefig('comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE - All visualizations saved!")
print("=" * 80)